In [24]:
import numpy as np
import matplotlib.pyplot as plt
import uproot
import os
from scipy.signal import find_peaks
import multiprocessing as mp

DIR = '/eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/'
SIM_MODE = 'aronly'
ADC_MODE = 'normal'
file = 'zzz_.root'

#file = uproot.open(DIR + '/' + file)["opdigiana"]
#file.keys()

In [ ]:
%matplotlib widget
y, x = file['event_1_opchannel_89_waveform_1;1'].to_numpy()
plt.figure()

plt.plot(y)

w = 300
y_smooth = np.convolve(y, np.ones(w), 'valid') / w

plt.figure()
plt.plot(y_smooth)

peaks, _ = find_peaks(y_smooth, height=0)
plt.plot(peaks, y_smooth[peaks], 'x')


time_resolution = np.diff(x)[0] # In microseconds

for i in range(20):
    baseline = np.mean(y[-(40 + i * 10) : - (20 + i * 10)])
    rms = np.std(y[-(40 + i * 10) : - (20 + i * 10)])
    if rms < 2:
        break
        
print(baseline, rms)

# SPE histogram
base_y = y - baseline
charge = []
for i in range(20):
    charge.append(np.sum(base_y[-(40 + i * 3) : - (37 + i * 3)]))


In [25]:
def get_charge_and_rms(x, y):    
    time_resolution = np.diff(x)[0] # In microseconds
    
    minim = 10000
    for i in range(15):
        baseline = np.mean(y[-(40 + i * 10) : - (20 + i * 10)])
        rms = np.std(y[-(40 + i * 10) : - (20 + i * 10)])
        if rms < minim:
            minim = rms

    # SPE histogram
    charge = []
    noise_charge = []
    base_y = y - baseline
    #for i in range(4):
     #   sub_charge = np.sum(base_y[-(10 + i * 6) : - (4 + i * 6)])
      #  charge.append(sub_charge)
    
    # Next SPE histogram... and charge in 1 us
    ticks = 1 / 0.016
    ticks_2 = ticks
    peaks = []
    random_peaks = []
    
    used_peak = 0
    if use_peak(base_y) and np.max(base_y) > 0 and np.argmax(base_y) < ticks:
        used_peak = 1
        peaks.append(np.max(base_y))
    
    if rms < 2 and np.abs(np.mean(base_y[0:10])) < 2 and np.abs(np.mean(base_y[-10:-1])) < 2 and np.max(base_y) > -90:
        # Random 1 us?
        #print(len(base_y))
        if len(base_y) > int(ticks):
            for i in range(10):
                r = np.random.randint(0, len(base_y) - int(ticks))
                selection = base_y[r : r + int(ticks)]
                charge.append(np.sum(selection))
                #random_peaks.append(np.max(selection))
                
                noise_charge.append(np.sum(base_y[-40:-10] * ticks / 30))
        else:
            print(len(base_y), ticks, ")))?I?")
    
    
    if rms < 2 and np.abs(np.mean(base_y[0:10])) < 2 and np.abs(np.mean(base_y[-10:-1])) < 2 and np.max(base_y) > -90:
        if len(base_y) > int(ticks):
            for i in range(10):
                r = np.random.randint(0, len(base_y) - int(ticks_2))
                selection = base_y[r : r + int(ticks_2)]
                random_peaks.append(np.max(selection))
        else:
            print(len(base_y), ticks, ")))?I?")
    
    return baseline, rms, charge, noise_charge, peaks, used_peak, random_peaks

In [26]:
def use_peak(y):
    
    pre_baseline = False
    if len(y) < 10:
        return False
    
    if np.abs(np.mean(y[0:10])) < 1:
        pre_baseline = True
    
    if not pre_baseline:
        return False
    
    w = 100
    y_smooth = np.convolve(y, np.ones(w), 'valid') / w

    peaks, _ = find_peaks(y_smooth, height=1)
    peaks = np.sort(peaks)
    
    if len(peaks) == 0:
        return False
    if len(peaks) < 4:
        return True

    ratio = peaks[-1]/peaks[-3]
    
    if ratio > 1.0:
        return True
    
    return False


In [27]:
def load_all_sn_events_chunky(limit=1, sigtype="BG", offset=0):
    
    if sigtype == "SN":
        dir = '/eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/'
    elif sigtype == "BG":
        dir = '/eos/project-e/ep-nu/pbarhama/sn_saves/prod_background_pds/Ar39GenInLAr/'
        
    directory = os.fsencode(dir)
    endswith = '_g4_detsim_{}_digi_hist.root'.format(SIM_MODE)
    
    if sigtype == "SN":
        if ADC_MODE == 'low':
            startswith = 'prodmarley_nue_dune10kt_vd_1x8x14_larger_lowADC'
        elif ADC_MODE == 'normal':
            startswith = 'prodmarley_nue_dune10kt_vd_1x8x14_larger'
    
    elif sigtype == "BG":
        if ADC_MODE == 'low':
            startswith = 'prodbg_radiological_dune10kt_vd_1x8x14_lowADC'
        elif ADC_MODE == 'normal':
            startswith = 'prodbg_radiological_dune10kt_vd_1x8x14'
        

    #print(endswith, startswith)
    
    file_names = []
    info_file_names = [] 
    i = 0

    for file in os.listdir(directory):
        filename = os.fsdecode(file)

        if filename.endswith(endswith) and filename.startswith(startswith):
            #if ADC_MODE == 'normal' and 'lowADC' in filename:
             #   continue

            if i < offset:
                i += 1
                continue
            
            i += 1
            if i > limit + offset:
                break

            filename = dir + filename
            file_names.append(filename)
    
    #digi_files = []

    #for file in file_names:
     #   digi_files.append(uproot.open(file)["opdigiana"])

    return file_names

In [28]:
def get_wf_data(num, d_file):
    print("{} {} \n".format(num, d_file))
    
    part_used_peaks = []
    part_charge = []
    part_noise_charge = []
    part_peaks = []
    part_random_peaks = []
    part_rms = []
    
    with uproot.open(d_file)["opdigiana"] as file:

        wf_lim = 20000
        wf_count = 0
        for i, key in enumerate(file.keys()):

            a = key.split("opchannel_", 1)[1]
            opid = a.split("_", 1)[0]

            if opid == "1":
                y, x = file[key].to_numpy()
                baseline, rms, charge, noise_charge, peaks, used_peak, random_peak = get_charge_and_rms(x[:-1], y)

                part_used_peaks.append(used_peak)
                part_charge.extend(charge)
                part_peaks.extend(peaks)
                part_noise_charge.extend(noise_charge)
                part_rms.append(rms)
                part_random_peaks.extend(random_peak)

                wf_count += 1
                if wf_count >= wf_lim:
                    break
                    
    return baseline, part_rms, part_charge, part_noise_charge, part_peaks, part_used_peaks, part_random_peaks

In [29]:
digi_files = load_all_sn_events_chunky(limit=500, sigtype="SN", offset=0)
print(len(digi_files))

all_charge = []
all_noise_charge = []
all_peaks = []
used_peaks = []
random_peaks = []
all_rms = []


with mp.Pool(mp.cpu_count()) as pool:
    results = pool.starmap(get_wf_data, zip(np.arange(len(digi_files)), digi_files))
#results = []
#for i, d_file in enumerate(digi_files):
#    results.append(get_wf_data(i, d_file))

for result in results:
    baseline, rms, charge, noise_charge, peaks, used_peak, random_peak = result
    used_peaks.extend(used_peak)
    all_charge.extend(charge)
    all_peaks.extend(peaks)
    all_noise_charge.extend(noise_charge)
    all_rms.extend(rms)
    random_peaks.extend(random_peak)
    

print(len(results))

450
16 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417774.481940067_g4_detsim_aronly_digi_hist.root 
64 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417790.321169165_g4_detsim_aronly_digi_hist.root 
48 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417785.034827685_g4_detsim_aronly_digi_hist.root 
8 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417765.914592473_g4_detsim_aronly_digi_hist.root 
56 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417787.752723727_g4_detsim_aronly_digi_hist.root 
32 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417779.100257811_g4_detsim_aronly_digi_hist.root 

72 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_

/cvmfs/sft.cern.ch/lcg/views/LCG_102swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3474: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,


105 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417805.418526208_g4_detsim_aronly_digi_hist.root 


/cvmfs/sft.cern.ch/lcg/views/LCG_102swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/numpy/core/_methods.py:189: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


/cvmfs/sft.cern.ch/lcg/views/LCG_102swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/numpy/core/_methods.py:264: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/cvmfs/sft.cern.ch/lcg/views/LCG_102swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/numpy/core/_methods.py:222: RuntimeWarning: invalid value encountered in true_divide
  arrmean = um.true_divide(arrmean, div, out=arrmean, casting='unsafe',
/cvmfs/sft.cern.ch/lcg/views/LCG_102swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/numpy/core/_methods.py:256: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


...

238 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417856.409016507_g4_detsim_aronly_digi_hist.root 

214 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417853.114575577_g4_detsim_aronly_digi_hist.root 

190 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417843.455574681_g4_detsim_aronly_digi_hist.root 

135 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417818.244114903_g4_detsim_aronly_digi_hist.root 

151 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417822.706436328_g4_detsim_aronly_digi_hist.root 

222 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417854.454021327_g4_detsim_aronly_digi_hist.root 

206 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dun


345 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417891.675757285_g4_detsim_aronly_digi_hist.root 

266 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417867.979796681_g4_detsim_aronly_digi_hist.root 

274 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417869.851373065_g4_detsim_aronly_digi_hist.root 

282 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417871.033912257_g4_detsim_aronly_digi_hist.root 

361 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417898.207065169_g4_detsim_aronly_digi_hist.root 

369 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417903.491298174_g4_detsim_aronly_digi_hist.root 

377 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_du


372 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417905.086490222_g4_detsim_aronly_digi_hist.root 

309 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417877.341147361_g4_detsim_aronly_digi_hist.root 

380 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417906.899602066_g4_detsim_aronly_digi_hist.root 

317 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417878.915875431_g4_detsim_aronly_digi_hist.root 

262 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417866.322379189_g4_detsim_aronly_digi_hist.root 

293 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417874.283654795_g4_detsim_aronly_digi_hist.root 

341 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_du


385 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417908.118534819_g4_detsim_aronly_digi_hist.root 

393 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417912.162128500_g4_detsim_aronly_digi_hist.root 

401 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417913.929600042_g4_detsim_aronly_digi_hist.root 

409 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417915.838997435_g4_detsim_aronly_digi_hist.root 

417 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417920.188200300_g4_detsim_aronly_digi_hist.root 

425 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417925.772476753_g4_detsim_aronly_digi_hist.root 

433 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_du


431 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417930.810991051_g4_detsim_aronly_digi_hist.root 

439 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668417932.515708549_g4_detsim_aronly_digi_hist.root 

447 /eos/project-e/ep-nu/pbarhama/sn_saves/prod_snnue_pds/prodmarley_nue_dune10kt_vd_1x8x14_larger_1668418008.746324789_g4_detsim_aronly_digi_hist.root 

450


In [30]:
%matplotlib widget
print(len(all_charge))
plt.figure(1)
plt.hist(all_charge, bins=np.linspace(-100, 500, 200), histtype='step')
plt.yscale('log')
plt.xlabel("ADC counts * ticks")

#plt.figure(2)
#plt.hist(all_noise_charge, bins=100, histtype='step')
#plt.yscale('log')

plt.figure(3)
plt.hist(all_peaks, bins=np.linspace(0, 80, 280), histtype='step')
plt.yscale('log')
plt.vlines(13, 0, 100, color="red")
plt.vlines(20, 0, 100, color="red")
plt.xlabel("ADC counts")
plt.title("All peaks")

plt.figure(4)
plt.hist(random_peaks, bins=np.linspace(0, 60, 250), histtype='step')
plt.yscale('log')
plt.xlabel("ADC counts")
plt.title("Random peaks")
#plt.xlim(left=7)

print(np.sum(used_peaks)/len(used_peaks))

plt.figure(5)
plt.hist(all_rms, bins=100)
plt.xlabel("RMS")

28070


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0.6578224776500639


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

Text(0.5, 0, 'RMS')

In [ ]:
digi_files = load_all_sn_events_chunky(limit=500, sigtype="SN", offset=0)
print(len(digi_files))